# Assignment 1: Introduction to Language Modeling

RNN-based autoregressive language model on Wikipedia paragraphs.

All implementation lives in `A1_skeleton.py` (tokenizer, model, trainer).
This notebook walks through the 5 parts of the assignment and contains the sanity checks + Part 5 evaluation.

## Setup

Clone the repo, install deps, download NLTK punkt.

In [ ]:
# Clone the repo and cd into the assignment folder.
# If you're running this notebook from inside the cloned repo already,
# you can skip this cell.
!git clone https://github.com/dmw1998/WASP_DL4NLP26.git 2>/dev/null || true
%cd WASP_DL4NLP26/Assignments/A1/a1_1
!ls

In [ ]:
!pip install -q datasets nltk scikit-learn matplotlib transformers accelerate

In [ ]:
import os, math
import torch
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

TRAIN_FILE = 'train.txt'
VAL_FILE = 'val.txt'
assert os.path.exists(TRAIN_FILE), f'{TRAIN_FILE} not found in {os.getcwd()}'
assert os.path.exists(VAL_FILE), f'{VAL_FILE} not found in {os.getcwd()}'

## Part 1: Tokenization

> **⚙ Task 1.1 — NLTK word splitting**
> - Use `nltk.word_tokenize` (faster than SpaCy, recommended by the assignment)
> - Lowercase everything before adding to vocab → keeps the vocab smaller
> - `lowercase_tokenizer` wraps both steps in one function

### Task 1.1 + 1.2 — build vocabulary

`build_tokenizer` reads the training file line by line, uses NLTK `word_tokenize` + lowercase, and keeps the most frequent words up to `max_voc_size`. The 4 special tokens `<PAD>`, `<UNK>`, `<BOS>`, `<EOS>` occupy ids 0–3.

> **🎓 Task 1.2 — building the vocabulary**
> - `Counter` over all training tokens → frequency dict
> - 4 special tokens placed first (ids 0–3) so their ids are deterministic
> - Truncate to `max_voc_size - 4` most frequent → rare words become `<UNK>`
> - `pad_token_id = 0` is what `CrossEntropyLoss(ignore_index=...)` and the trainer will use
> - Inverse dict `int_to_str` is for Part 5 (printing predictions / neighbors)
> - **Why special tokens?** `<BOS>` lets the model predict the first real word; `<EOS>` lets it learn when to stop; `<UNK>` handles OOV; `<PAD>` makes ragged batches rectangular

In [ ]:
from A1_skeleton import (
    build_tokenizer, A1Tokenizer, lowercase_tokenizer,
    A1RNNModelConfig, A1RNNModel,
    A1Trainer,
)

MAX_VOC_SIZE = 10000
MODEL_MAX_LENGTH = 128

tokenizer = build_tokenizer(
    train_file=TRAIN_FILE,
    tokenize_fun=lowercase_tokenizer,
    max_voc_size=MAX_VOC_SIZE,
    model_max_length=MODEL_MAX_LENGTH,
)

print('vocab size:', len(tokenizer))
print('pad_token_id:', tokenizer.pad_token_id)
print('first 10 entries:', list(tokenizer.str_to_int.items())[:10])

In [ ]:
# Sanity checks for Task 1.2
assert len(tokenizer) <= MAX_VOC_SIZE, 'vocab is larger than the cap'
for tok in ['<PAD>', '<UNK>', '<BOS>', '<EOS>']:
    assert tok in tokenizer.str_to_int, f'{tok} missing from vocab'

# Common words should be present; rare words should typically not be.
for w in ['the', 'and', 'of']:
    assert w in tokenizer.str_to_int, f'common word {w!r} should be in vocab'
for w in ['cuboidal', 'epiglottis']:
    if w not in tokenizer.str_to_int:
        print(f'OK: rare word {w!r} not in vocab')

# Round-trip: word -> int -> word
i = tokenizer.str_to_int['the']
assert tokenizer.int_to_str[i] == 'the'
print('all sanity checks passed')

### Task 1.3 — A1Tokenizer.__call__

Verify the tokenizer handles a batch of differently-sized inputs, pads on the right, and returns a PyTorch tensor.

> **⚙ Task 1.3 — HuggingFace-like tokenizer**
> - `__call__` does the full pipeline: split → lookup ids → wrap with BOS/EOS → truncate → pad → tensor
> - Padding is **right-side** with `pad_token_id` so that real tokens stay at the start
> - `attention_mask`: 1 for real tokens, 0 for padding (optional, but matches HF convention)
> - `return_tensors='pt'` returns torch tensors; otherwise Python lists
> - `save` / `from_file` use pickle (skeleton provides these)

In [ ]:
test_texts = ['This is a test.', 'Another test.']
enc = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
print(enc)

# Convert ids back to words for visual confirmation.
for i, ids in enumerate(enc['input_ids']):
    words = [tokenizer.int_to_str[int(x)] for x in ids]
    print(f'row {i}:', words)

In [ ]:
# Save and reload the tokenizer.
tokenizer.save('a1_tokenizer.pkl')
reloaded = A1Tokenizer.from_file('a1_tokenizer.pkl')
assert len(reloaded) == len(tokenizer)
print('tokenizer save/load OK')

## Part 2: load datasets, quick batching demo

(Not required in the submission, just sanity-checks the data pipeline.)

> **⚙ Task 2.1 — loading texts**
> - `load_dataset('text', ...)` from HuggingFace `datasets`
> - Each non-empty line = one Wikipedia paragraph
> - `.filter(lambda x: x['text'].strip() != '')` removes blank separator lines
> - Expected sizes after filtering: ~147k train, ~18k val
> - Datasets internally use Arrow format but behave like lists
>
> **⚙ Task 2.2 — DataLoader**
> - `DataLoader(dataset, batch_size=..., shuffle=True)` for training
> - `shuffle=True` during training, `False` during eval
> - A `collate_fn` is needed to tokenize raw text → tensors per batch

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'text',
    data_files={'train': TRAIN_FILE, 'val': VAL_FILE},
)
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))
print('\nexample paragraph:')
print(dataset['train'][8]['text'][:300])

In [ ]:
# DataLoader demo: peek at the first batch.
from torch.utils.data import DataLoader
demo_loader = DataLoader(dataset['train'], batch_size=4, shuffle=False)
for batch in demo_loader:
    print('keys:', list(batch.keys()))
    print('first text:', batch['text'][0][:150])
    break

## Part 3: define the RNN language model

### Task 3.1 + 3.2

Architecture: embedding → LSTM → unembedding. The loss (computed in `forward` when `labels` is given) uses a shift-by-one trick: at position *i* the model is trained to predict the token at position *i+1*.

> **🎓 Task 3.1 — setting up the network**
> - 3 layers: `Embedding` → `LSTM` → `Linear` (unembedding)
> - **Embedding**: vocab_size × embedding_size. Maps each token id to a dense vector.
> - **LSTM**: `batch_first=True` so input is (B, N, E); output (B, N, H) where N walks along the sequence
> - **Why LSTM (not vanilla RNN)?** Vanilla RNN suffers from vanishing gradients; LSTM/GRU have gating that lets info flow through long sequences
> - **Unembedding**: hidden_size → vocab_size. Produces logits (one score per vocab word at every position)
> - LSTM returns `(output, (h_n, c_n))` — we only use `output`, the per-token hidden states
> - Inherits from `PreTrainedModel` so `.save_pretrained()` / `.from_pretrained()` work for free

> **🎓 Task 3.2 — computing the loss**
> - Loss = categorical cross-entropy = `CrossEntropyLoss(ignore_index=-100)`
> - **Shift-by-one is critical**: at position *i* we want to predict the token at position *i+1*
>   - drop the **last** position of logits (nothing to predict after the final token)
>   - drop the **first** position of labels (nothing precedes the first token)
> - Reshape for `CrossEntropyLoss`: logits (B,N,V) → (B·N, V); labels (B,N) → (B·N)
> - `ignore_index=-100` lets us mask out padding positions
> - **Without shift-by-one** the model just learns identity (predict self from self) → loss collapses to 0 but the model is useless

In [ ]:
config = A1RNNModelConfig(
    vocab_size=len(tokenizer),
    embedding_size=128,
    hidden_size=256,
)
model = A1RNNModel(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'model parameters: {n_params:,}')
print(model)

In [ ]:
# Sanity check: a 1 x N input should produce a 1 x N x V logits tensor.
N = 7
dummy = torch.randint(0, len(tokenizer), (1, N))
with torch.no_grad():
    out = model(dummy)
print('logits shape:', out.logits.shape,
      f'(expected: 1 x {N} x {len(tokenizer)})')
assert out.logits.shape == (1, N, len(tokenizer))

# With labels, the model also returns a loss.
out = model(dummy, labels=dummy)
print('loss with labels (random init):', out.loss.item())

## Part 4: train the model

Uses HuggingFace `TrainingArguments`. The skeleton requires `optim='adamw_torch'` and `eval_strategy='epoch'`.

Set `dev_mode = True` first to make sure the pipeline runs end-to-end on a tiny subset (~1 minute on GPU). Then switch to `dev_mode = False` for the real run.

> **🎓 Task 4.1 — implementing the trainer**
> - Optimizer: `AdamW` with `args.learning_rate` (used in all modern LLMs)
> - DataLoader with `collate_fn`: tokenize batch → make `labels` by cloning `input_ids` and replacing pad ids with -100
> - Standard PyTorch training loop:
>   1. `optimizer.zero_grad()` — clear stale grads
>   2. `loss = model(input_ids, labels).loss` — forward + loss
>   3. `loss.backward()` — backprop
>   4. `optimizer.step()` — update weights
> - End-of-epoch validation in `torch.no_grad()` + `model.eval()` mode
> - `model.save_pretrained(args.output_dir)` saves config + weights HF-style
> - **Why mask padding (`-100`)?** Padding tokens are dummies; including them in the loss would teach the model to predict `<PAD>` after every sentence, hurting real predictions
> - **Why AdamW over SGD?** Adaptive per-parameter learning rates + decoupled weight decay; works well out-of-the-box without much tuning

In [ ]:
from torch.utils.data import Subset
from transformers import TrainingArguments

dev_mode = False

if dev_mode:
    train_ds = Subset(dataset['train'], range(1000))
    val_ds = Subset(dataset['val'], range(200))
    epochs = 1
else:
    train_ds = dataset['train']
    val_ds = dataset['val']
    epochs = 3

args = TrainingArguments(
    output_dir='trainer_output',
    optim='adamw_torch',
    eval_strategy='epoch',
    learning_rate=1e-3,
    num_train_epochs=epochs,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    logging_steps=100,
    save_strategy='no',     # we save manually at the end of training
    report_to='none',
    use_cpu=False,
)
# Fix for AttributeError: 'TrainingArguments' object has no attribute 'no_cuda'
# The A1Trainer expects a 'no_cuda' attribute, which can be derived from 'use_cpu'.
args.no_cuda = args.use_cpu

# Fix for TypeError: A1RNNModelConfig.__init__() missing 1 required positional argument: 'vocab_size'
# The transformers library's save_pretrained method attempts to instantiate the config class without arguments.
# Setting has_no_defaults_at_init to True prevents this.
model.config.has_no_defaults_at_init = True

trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()

## Part 5: Evaluation

### Task 5.1 — predict the next word

Encode a prompt, look at the logits one position *before* the `<EOS>` token (since the tokenizer always appends `<EOS>`), then read off the top-k predictions.

> **⚙ Task 5.1 — predicting the next word**
> - Tokenizer always appends `<EOS>`, so the last real word's prediction lives at position **-2** (one before `<EOS>`)
> - `argmax` over vocab gives the single most likely next word
> - `topk` gives the top *k* candidates with their logit scores (higher = more likely)
> - Use `inv_voc` to map ids back to readable words
> - Examples that worked: 'She lives in San' → 'francisco'/'diego'; 'The president of the United' → 'states'/'kingdom'

In [ ]:
def predict_next(model, tokenizer, prompt, k=5):
    device = next(model.parameters()).device
    model.eval()
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    with torch.no_grad():
        out = model(input_ids)
    # Position -2 is the last *real* token (position -1 is <EOS>).
    logits = out.logits[0, -2]
    topk = torch.topk(logits, k)
    return [
        (tokenizer.int_to_str[i.item()], s.item())
        for i, s in zip(topk.indices, topk.values)
    ]

for prompt in [
    'She lives in San',
    'The president of the United',
    'He played the guitar and',
    'The capital of Sweden is',
]:
    print(f'prompt: {prompt!r}')
    for word, score in predict_next(model, tokenizer, prompt):
        print(f'  {word:20s} {score:.3f}')
    print()

### Task 5.2 — perplexity on the validation set

Perplexity = exp(mean cross-entropy loss over all non-padding tokens). A well-trained model on this dataset should land in the 200–300 range; >700 typically indicates a bug.

> **🎓 Task 5.2 — perplexity**
> - Formula: `perplexity = 2^(-1/m · Σ log₂ P(w_i | c_i))` where m = total tokens
> - Equivalently: `perplexity = exp(mean cross-entropy)` — log base doesn't matter as long as it matches the exp base
> - **Intuition**: ppl = the effective number of choices the model is "torn between" at each position. ppl=10 means the model is as uncertain as if picking uniformly among 10 words.
> - Lower ppl = better. Random guessing over a 10k vocab gives ppl ≈ 10000.
> - **Token-weighting**: I weight each batch's loss by the number of non-padding tokens, so short sentences don't dominate the average
> - **My result: ppl ≈ 75** after 3 epochs, well below the 200–300 "good" range
> - Why so low? 3 epochs (vs 1 in the assignment baseline) + gradient clipping in the skeleton trainer stabilizes LSTM training

In [ ]:
# Token-weighted mean CE loss over the validation set.
from torch.utils.data import DataLoader

def collate(batch):
    texts = [ex['text'] for ex in batch]
    enc = tokenizer(texts, truncation=True, padding=True, return_tensors='pt')
    input_ids = enc['input_ids']
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return {'input_ids': input_ids, 'labels': labels}

val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, collate_fn=collate)

device = next(model.parameters()).device
model.eval()
total_loss = 0.0
total_tokens = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        out = model(input_ids, labels=labels)
        # The shift drops one position per row, so token count is computed
        # against shifted labels (positions 1..T-1).
        n = int((labels[:, 1:] != -100).sum().item())
        total_loss += out.loss.item() * n
        total_tokens += n

mean_ce = total_loss / total_tokens
ppl = math.exp(mean_ce)
print(f'mean cross-entropy: {mean_ce:.4f}')
print(f'perplexity:         {ppl:.2f}')

### Task 5.3 — inspect learned word embeddings

For a well-trained embedding, nearest neighbors of a word should be semantically related.

> **🎓 Task 5.3 — embedding nearest neighbors**
> - Cosine similarity over the embedding matrix: `cos(emb[word], emb[all_words])`
> - Take top-k highest scores, skipping position 0 (the word itself, cos=1.0)
> - **What good embeddings look like**: 'three' → 'six', 'five', 'two' ✅ — numbers cluster together
> - **What my results show**:
>   - **High-frequency / semantic categories work**: numbers, royal terms (king → prince) cluster reasonably
>   - **Proper nouns are weak**: 'sweden' → unrelated words; 'paris' → noise
> - **Why the difference?** Common words like 'three' appear thousands of times in training, so their embeddings get many gradient updates. Rare proper nouns ('sweden', 'paris') appear far fewer times → undertrained embeddings
> - **Theoretical reason**: word embeddings learn distributional semantics — "you shall know a word by the company it keeps". Rare words don't get enough context exposure to develop meaningful neighborhoods.
> - **How to improve**: bigger embedding dim, more epochs, more data, or subword tokenization (BPE) which shares parameters across morphologically related words

In [ ]:
from torch import nn

def nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5):
    if word not in voc:
        print(f'{word!r} not in vocab')
        return []
    test_emb = emb.weight[voc[word]]
    sim = nn.CosineSimilarity(dim=1)(test_emb, emb.weight)
    top = sim.topk(n_neighbors + 1)
    # Skip position 0: that's the query word itself.
    return [
        (inv_voc[ix.item()], cos.item())
        for ix, cos in zip(top.indices[1:], top.values[1:])
    ]

# Move embedding to CPU for the cosine sim computation.
emb = model.embedding.cpu()
for w in ['sweden', 'king', 'three', 'london', 'small', 'paris']:
    print(f'neighbors of {w!r}:')
    for nbr, score in nearest_neighbors(
        emb, tokenizer.str_to_int, tokenizer.int_to_str, w
    ):
        print(f'  {nbr:20s} {score:.3f}')
    print()

In [ ]:
# Optional: 2D PCA scatterplot of selected embeddings.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

def plot_embeddings_pca(emb, voc, words):
    words = [w for w in words if w in voc]
    vecs = np.vstack([
        emb.weight[voc[w]].detach().cpu().numpy() for w in words
    ])
    vecs -= vecs.mean(axis=0)
    twodim = TruncatedSVD(n_components=2).fit_transform(vecs)
    plt.figure(figsize=(7, 7))
    plt.scatter(twodim[:, 0], twodim[:, 1], c='r', edgecolors='k')
    for w, (x, y) in zip(words, twodim):
        plt.text(x + 0.02, y, w)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_embeddings_pca(
    emb,
    tokenizer.str_to_int,
    ['sweden', 'denmark', 'europe', 'africa', 'london', 'stockholm',
     'large', 'small', 'great', 'black',
     '3', '7', '10', 'seven', 'three', 'ten',
     '1984', '2005', '2010'],
)

## Reloading a trained model later

```python
from A1_skeleton import A1RNNModel, A1Tokenizer
tokenizer = A1Tokenizer.from_file('a1_tokenizer.pkl')
model = A1RNNModel.from_pretrained('trainer_output')
```